In [1]:
import os
import weaviate
import weaviate.classes.config as wvc

from dotenv import load_dotenv
from openai import OpenAI

import requests
from bs4 import BeautifulSoup
import tiktoken

In [2]:
load_dotenv(r'.env')

True

In [3]:
def get_content(url):
    response = requests.get(url)

    soup = BeautifulSoup(response.text, 'html.parser')

    for script_or_style in soup(["script", "style", "header", "footer", "nav"]):
        script_or_style.decompose()
        

    text = soup.get_text(separator="\n")

    lines = [line.strip() for line in text.splitlines() if line.strip()]
    clean_text = " ".join(lines)
    return clean_text

In [4]:
def seed_chunks_by_tokens(text, model_name="gpt-4o", chunk_size=500, chunk_overlap=50):

    encoding = tiktoken.encoding_for_model(model_name)
    
    all_tokens = encoding.encode(text)
    
    chunks = []

    for i in range(0, len(all_tokens), chunk_size - chunk_overlap):

        chunk_tokens = all_tokens[i : i + chunk_size]
        chunk_text = encoding.decode(chunk_tokens)
        chunks.append(chunk_text)
        
    return chunks


In [5]:
chunk_size = 100
chunk_overlap = 20
chunks = []
url1 = "https://www.azfamily.com/2024/07/01/evacuation-orders-lifted-boulder-view-fire-near-scottsdale-63-contained/"
url2 = "https://www.kjzz.org/news/2024-06-28/60-forced-to-evacuate-from-boulder-view-fire-in-north-scottsdale"

content1 = get_content(url1)
chunks1 = seed_chunks_by_tokens(content1, chunk_size=chunk_size, chunk_overlap=chunk_overlap)

content2 = get_content(url2)
chunks2 = seed_chunks_by_tokens(content2, chunk_size=chunk_size, chunk_overlap=chunk_overlap)



for c in chunks1:
    chunks.append({'content': c, "source_url": url1})

for c in chunks2:
    chunks.append({'content': c, "source_url": url2})


In [6]:
print(chunks[0])

{'content': 'Boulder View Fire 95% contained near Scottsdale Skip to content Share Add Us On Google Add as a preferred source on Google SCOTTSDALE, AZ (AZFamily) — The Boulder View Fire, which has burned thousands of acres northwest of Scottsdale, is now 95% contained as of July 4. The Boulder View Fire first sparked on June 27 and has burned 3,711 acres. The fire hasn’t grown in acreage for days. The #BoulderViewFire is', 'source_url': 'https://www.azfamily.com/2024/07/01/evacuation-orders-lifted-boulder-view-fire-near-scottsdale-63-contained/'}


In [7]:
client = weaviate.connect_to_local(
    headers={"X-OpenAI-Api-Key": os.getenv("OPENAI_API_KEY")}
)
client.connect()

In [8]:
collection_name = "NewsChunk"

# client.collections.delete(collection_name)

if not client.collections.exists(collection_name):
    client.collections.create(
        name=collection_name,
        vector_config=[wvc.Configure.Vectors.text2vec_openai(
            name='core', 
            source_properties=['content'],
            model='text-embedding-3-large',
            dimensions=1024
        )],
        properties=[
            wvc.Property(name="content", 
                         data_type=wvc.DataType.TEXT),
            wvc.Property(name="source_url", 
                         data_type=wvc.DataType.TEXT)
        ]
    )
    news_chunks = client.collections.get(collection_name)
    with news_chunks.batch.dynamic() as batch:
        for c in chunks:
            batch.add_object(
                properties={
                    "content": c['content'],
                    "source_url": c['source_url']
                }
            )
    print(f"successfully stored {len(chunks)} data points")
news_chunks = client.collections.use(collection_name)

In [9]:
def RAG_Q_A(question, vector_database, LLM_clinet, model_name):
    response = vector_database.query.near_text(
    query=question,
    limit=3)

    retrieved_content = ''
    for obj in response.objects:
        retrieved_content += obj.properties['content'] + '\n'
        # print(retrieved_content)
        
    augmented_question = augmented_question_template.format(question=question, retrieved_content=retrieved_content)

    response = LLM_client.responses.create(
        model = model_name,
        input = augmented_question
        )

    return response.output_text, retrieved_content

In [10]:
augmented_question_template = '''
You will need to answer my question based on the information I give you,

This is the question:
{question}

This is the background information:
{retrieved_content}
'''

LLM_client = OpenAI()
model_name = 'gpt-4.1-mini'

In [11]:
question = "What roads or areas were affected (closed or restricted) during the fire, and which of them were later reopened?"
answer, retrieved_content = RAG_Q_A(question, news_chunks, LLM_client, model_name)

In [12]:
print(answer)

Based on the information provided:

**Roads or areas affected (closed or restricted) during the fire:**
- The north part of the McDowell Sonoran Preserve, including all trails and access points north of Dynamite Boulevard, was closed until further notice.
- Residents in the evacuation zone, specifically from 136th Street to Box Bar Road and Rio Verde Road to Dove Valley, were told to evacuate.
- Evacuation orders also affected around 60 homes.

**Roads or areas later reopened:**
- All trailheads in the north part of the McDowell Sonoran Preserve were reopened on Monday morning.
- Bartlett Lake Dam Road and Horseshoe Dam Road were reopened on Monday morning.
- Evacuation orders for the residents from 136th Street to Box Bar Road and Rio Verde Road to Dove Valley were lifted on Saturday afternoon, allowing them to return to "READY" status.

In summary, the north part of McDowell Sonoran Preserve trails and access points, Bartlett Lake Dam Road, and Horseshoe Dam Road were closed during t

In [13]:
print(retrieved_content)

 out of everyone’s way and help where I can,” said Engelker. The City of Scottsdale notified homeowners that the north part of the McDowell Sonoran Preserve, including all trails and access points north of Dynamite Boulevard, would be closed until further notice. All trailheads were reopened Monday morning. Bartlett Lake Dam Road and Horseshoe Dam Road were reopened Monday morning. Tiffany Davila, spokeswoman with the Department of Forestry and Fire Management, says around 60 homes were evacuated. The fire also
 statuses for those near the fire. This means that residents in the evacuation zone are returning to “READY” status as crews made progress on the fire. On Friday, residents in the area from 136th Street to Box Bar Road and Rio Verde Road to Dove Valley were told to evacuate as the fire grew closer to their neighborhood. Those evacuation orders were lifted on Saturday afternoon. “We were woken up by a police helicopter with spotlights in our area on the radio asking everybody to 

In [14]:
question = "At the beginning of the wildfire (June 28), where did evacuations occur and how many people (or homes) were evacuated?"
answer, retrieved_content = RAG_Q_A(question, news_chunks, LLM_client, model_name)

In [15]:
print(answer)

At the beginning of the wildfire on June 28, evacuations occurred in part of the Vista Verde community, which is a subdivision on the edge of the Tonto National Forest. Approximately 60 homes were evacuated, with around 60 people having left their homes by Friday morning after Maricopa County emergency personnel ordered the evacuations. Additionally, residents between 136th Street to Box Bar Road and Rio Verde to Dove Valley roads were advised to be prepared for evacuation.


In [16]:
print(retrieved_content)

 with the Department of Forestry and Fire Management, says around 60 homes were evacuated. The fire also threatened power lines in the area. “It’s very uneasy knowing that anything could happen at any moment. The wind shifts and fire was moving in one direction last night and then it shifted in the middle of the night, so here we are,” said Engelker. On the night it began, the southeast end of the fire grew significantly, prompting evacuations for part of the Vista Verde community. “
 60 people had left their homes by Friday morning after Maricopa County emergency personnel ordered evacuations for the subdivision on the edge of the Tonto National Forest. The Maricopa County Department of Emergency Management on Friday expanded the list of who needs to get set to evacuate. As of Friday morning, residents between 136th Street to Box Bar Road and Rio Verde to Dove Valley roads should be prepared. Fire officials said they were investigating exactly what sparked the blaze. Nearly 200 firefi

In [20]:
f1_path = r"/home/huihai/Huihai/interview/interviewRAG/graphrag/input/news1.txt"
f2_path = r"/home/huihai/Huihai/interview/interviewRAG/graphrag/input/news2.txt"

with open(f1_path, "w") as txt_file1:
    txt_file1.write(content1)

with open(f2_path, "w") as txt_file2:
    txt_file2.write(content2)

In [21]:
import pandas as pd

In [50]:
entities = pd.read_parquet(r"/home/huihai/Huihai/interview/interviewRAG/graphrag/output/entities.parquet")
relationships = pd.read_parquet(r"/home/huihai/Huihai/interview/interviewRAG/graphrag/output/relationships.parquet")
communities = pd.read_parquet(r"/home/huihai/Huihai/interview/interviewRAG/graphrag/output/communities.parquet")
communities_reports = pd.read_parquet(r"/home/huihai/Huihai/interview/interviewRAG/graphrag/output/community_reports.parquet")

In [51]:
entities['description'][32]

'Cave Creek Road is located near the Boulder View Fire.'

In [52]:
entities

,id,human_readable_id,title,type,description,text_unit_ids,frequency,degree,x,y
0,0e319622-6b8b-4ca3-8cd3-2af4e4a56b06,0,BOULDER VIEW FIRE,FIRE_EVENT,The Boulder View Fire is a significant wildfir...,[e6d6debbae3b1b433e35944390c1e13f0a90b7715b5d7...,2,28,0.0,0.0
1,1af61b5a-9518-46df-bbcc-4887dfd55988,1,NORTH SCOTTSDALE,FIRE_LOCATION,North Scottsdale is the geographic area where ...,[e6d6debbae3b1b433e35944390c1e13f0a90b7715b5d7...,1,1,0.0,0.0
2,7b1355ae-d180-4775-b670-1ba36a363cec,2,HUMAN-CAUSED,FIRE_CAUSE,The Boulder View Fire is identified as a human...,[e6d6debbae3b1b433e35944390c1e13f0a90b7715b5d7...,2,1,0.0,0.0
3,ef2bb11a-ab1d-4a64-bf92-f46877fac62c,3,STATUS_JUNE28_1146,STATUS,"As of June 28, 2024, at 11:46 AM MST, the Boul...",[e6d6debbae3b1b433e35944390c1e13f0a90b7715b5d7...,1,2,0.0,0.0
4,e175825a-2a24-4039-9a13-e1e7c6d3105c,4,3711 ACRES,BURNED_AREA,"As of June 28, 2024, the Boulder View Fire has...",[e6d6debbae3b1b433e35944390c1e13f0a90b7715b5d7...,2,1,0.0,0.0
5,48a955f3-a3ae-41f5-a994-b70c197415af,5,60 HOMES EVACUATED,EVACUATION_DETAIL,"On June 27, 2024, evacuation orders were issue...",[e6d6debbae3b1b433e35944390c1e13f0a90b7715b5d7...,2,1,0.0,0.0
6,b1f66f3c-b438-4691-84ca-28015208292c,6,270 FIREFIGHTERS,EMERGENCY_RESPONSE,Nearly 270 firefighters are actively battling ...,[e6d6debbae3b1b433e35944390c1e13f0a90b7715b5d7...,1,1,0.0,0.0
7,095d502d-57c9-46c6-9cba-658a57c130df,7,NO INJURIES REPORTED,PUBLIC_SENTIMENT,"As of the latest reports, no injuries have bee...",[e6d6debbae3b1b433e35944390c1e13f0a90b7715b5d7...,1,1,0.0,0.0
8,4796c73c-4422-4ddc-9086-b86f9234c483,8,ZERO PERCENT CONTAINMENT,CONTAINMENT_STATUS,"As of June 29, 2024, the Boulder View Fire is ...",[e6d6debbae3b1b433e35944390c1e13f0a90b7715b5d7...,1,0,NaN,NaN
9,d520edab-5b5a-4e6e-9f9a-dbb0e551e4bb,9,GO TO SET,EVACUATION_DETAIL,Evacuation orders were lowered from “GO” to “S...,[e6d6debbae3b1b433e35944390c1e13f0a90b7715b5d7...,1,1,0.0,0.0


In [28]:
entities[entities['type'] == "EVENT"]

,id,human_readable_id,title,type,description,text_unit_ids,frequency,degree,x,y
0,048db1c0-12c9-4496-8f9f-a2246f80307b,0,BOULDER VIEW FIRE,EVENT,The Boulder View Fire is a significant wildfir...,[e6d6debbae3b1b433e35944390c1e13f0a90b7715b5d7...,2,14,0.0,0.0
7,e1d8b5bd-2827-42b4-a3bc-c492e0577cd9,7,WILDCAT FIRE,EVENT,The Wildcat Fire is a previous wildfire that m...,[e6d6debbae3b1b433e35944390c1e13f0a90b7715b5d7...,1,1,0.0,0.0
17,b018d688-76ac-4f7b-8618-5df5284560cf,17,WILDFIRES,EVENT,Wildfires are large uncontrolled fires that ar...,[fc9c4bed00823db3f384e37e47892d5b31a40188d14b6...,1,6,0.0,0.0
34,85257bdd-38c3-4e99-9fa1-57752412986a,34,WOMEN'S FINAL FOUR,EVENT,The Women's Final Four is a championship weeke...,[364e4088f0a43d3aec204b2e79f94a5686f99429496a9...,1,1,0.0,0.0
36,94fe4991-f0b5-4449-b931-f1cfaffb0dd2,36,NATIONAL ANTHEM,EVENT,A performance of the national anthem at the D-...,[364e4088f0a43d3aec204b2e79f94a5686f99429496a9...,1,1,0.0,0.0
38,8c07f5fb-2a87-45d6-beb5-26f15484cae0,38,D-BACKS HOME OPENER,EVENT,The D-backs home opener is a significant event...,[364e4088f0a43d3aec204b2e79f94a5686f99429496a9...,1,1,0.0,0.0
39,f93a9a32-ff96-4a8f-90b6-23f8a5e933e3,39,HAZARDOUS AIR QUALITY,EVENT,Hazardous air quality is an environmental even...,[364e4088f0a43d3aec204b2e79f94a5686f99429496a9...,1,1,0.0,0.0


In [31]:
relationships['description'][0]

'The Boulder View Fire is a significant wildfire located near Scottsdale, Arizona, directly impacting the city and its residents. The fire has prompted evacuations and necessitated emergency responses to ensure the safety of those in the affected areas. As the situation develops, local authorities are actively managing the fire and coordinating efforts to protect the community from its effects. The ongoing response highlights the urgency of the situation and the commitment of emergency services to safeguard the residents of Scottsdale during this challenging time.'

In [53]:
relationships

,id,human_readable_id,source,target,description,weight,combined_degree,text_unit_ids
0,271d2fef-50cd-43a8-91b6-da5631cfb391,0,BOULDER VIEW FIRE,NORTH SCOTTSDALE,The Boulder View Fire is located in North Scot...,9.0,29,[e6d6debbae3b1b433e35944390c1e13f0a90b7715b5d7...
1,a38dff2d-701f-4e10-8210-58d16f58e187,1,BOULDER VIEW FIRE,HUMAN-CAUSED,The Boulder View Fire is an incident that is b...,17.0,29,[e6d6debbae3b1b433e35944390c1e13f0a90b7715b5d7...
2,e17b193c-fc4f-439a-8277-4851e33639a1,2,BOULDER VIEW FIRE,STATUS_JUNE28_1146,The status of the Boulder View Fire as of June...,7.0,30,[e6d6debbae3b1b433e35944390c1e13f0a90b7715b5d7...
3,e99ab2e0-e092-4aba-82a4-90991acbeee4,3,BOULDER VIEW FIRE,3711 ACRES,The Boulder View Fire has significantly impact...,16.0,29,[e6d6debbae3b1b433e35944390c1e13f0a90b7715b5d7...
4,d4cad18b-98bd-42db-b8a7-ac91298eb1b5,4,BOULDER VIEW FIRE,60 HOMES EVACUATED,The Boulder View Fire prompted the evacuation ...,16.0,29,[e6d6debbae3b1b433e35944390c1e13f0a90b7715b5d7...
5,c5704131-384b-46d1-adc9-5754a922c7d6,5,BOULDER VIEW FIRE,270 FIREFIGHTERS,The presence of nearly 270 firefighters is a d...,9.0,29,[e6d6debbae3b1b433e35944390c1e13f0a90b7715b5d7...
6,991e62c7-8bf7-42f1-8b51-d816033fc5b4,6,BOULDER VIEW FIRE,NO INJURIES REPORTED,The report of no injuries related to the Bould...,1.0,29,[e6d6debbae3b1b433e35944390c1e13f0a90b7715b5d7...
7,5143f501-b26f-4eba-b216-5d7122d63821,7,BOULDER VIEW FIRE,STATUS_JUNE29_2024,The status of the Boulder View Fire as of June...,7.0,29,[e6d6debbae3b1b433e35944390c1e13f0a90b7715b5d7...
8,83f56856-2df5-4558-b6fd-33b6328d7d12,8,BOULDER VIEW FIRE,MCDOWELL SONORAN PRESERVE,The McDowell Sonoran Preserve has been closed ...,7.0,29,[e6d6debbae3b1b433e35944390c1e13f0a90b7715b5d7...
9,a623a585-ff3f-4523-a885-f7c7cc017638,9,BOULDER VIEW FIRE,CAVE CREEK,Evacuation shelters for large animals were est...,6.0,29,[e6d6debbae3b1b433e35944390c1e13f0a90b7715b5d7...


In [34]:
communities

,id,human_readable_id,community,level,parent,children,title,entity_ids,relationship_ids,text_unit_ids,period,size
0,fcd39395-29f1-4936-a7a8-d2feca4d9eb7,0,0,0,-1,"[2, 3]",Community 0,"[b7785036-c474-414f-a234-968d8dacfacc, 048db1c...","[0b7415ee-168d-46be-9481-68fd7de49cc6, 0fefc66...",[d041353774cdf4d3b2fc64b76a3e20e5f21b950f2093b...,2026-03-30,15
1,08367477-7640-4324-b863-fff48bf23fc8,1,1,0,-1,[],Community 1,"[8efd67bd-8b7b-499a-8206-3560dc7ef387, ed8405c...","[08d529b0-bc94-4cc8-946a-a39fc78f91cd, 64e1eca...",[e6d6debbae3b1b433e35944390c1e13f0a90b7715b5d7...,2026-03-30,3
2,40a32584-53f5-42fc-90d7-d1a6c971d74c,2,2,1,0,[],Community 2,"[048db1c0-12c9-4496-8f9f-a2246f80307b, 3e520ae...","[0b7415ee-168d-46be-9481-68fd7de49cc6, 0fefc66...",[d041353774cdf4d3b2fc64b76a3e20e5f21b950f2093b...,2026-03-30,13
3,693f540a-8fc8-4e24-9ee3-0bae3913e410,3,3,1,0,[],Community 3,"[b7785036-c474-414f-a234-968d8dacfacc, bc29c07...","[8011211c-85d2-40f8-a7f0-268354bcabff, afaac08...",[d041353774cdf4d3b2fc64b76a3e20e5f21b950f2093b...,2026-03-30,2
